# Training Experiments

Now it is time to put everything together and train a few small language models on a pretaining dataset.

**What we will do:**
1. Train a transformer LM on the TinyStories dataset.
2. Generate samples and evaluate perplexity using the trained transformer LM.
3. Experiment with training on a sample of the OpenWebText dataset.

**Datasets:**
We will need the following two datasets in this notebook:
- TinyStories.
- OpenWebText (sample).

If you haven't yet downloaded these datasets, refer to the download instructions in the earlier tokenization notebook.

*Note:* This notebook may require some of the functions and classes you've implemented in previous notebooks. Feel free to copy them over where appropriate.

In [ ]:
import re
import regex
import numpy as np
import torch
import torch.nn as nn
from einops import rearrange, einsum

In [ ]:
seed = 42
np.random.seed(seed)
torch.manual_seed(seed)

In [ ]:
def get_device():
    if torch.cuda.is_available():
        return 'cuda'
    if torch.backends.mps.is_available():
        return 'mps'
    return 'cpu'

device = get_device()
print(f'device = {device}')

## How to Run Experiments

The best way to understand the rationale behind the architectural components of a transformer is to actually modify it and run it yourself. There is no substitute for hands-on experience.

To this end, it's important to be able to experiment *quickly, consistently, and keep records of* what you did. To experiment quickly, we will be running many experiments on a small scale model (17M parameters) and simple dataset (TinyStories). To do things consistently, we will ablate components and vary hyperparameters in a systematic way, and to keep records we will log our experiments and learning curves in each experiment.

To log the loss curves, make sure to periodically evaluate validation losses and record both the number of steps and wallclock times. You might find logging infrastructure such as Weights and Biases helpful.

**Experiment logging:** For your training and evaluation code, try to create an experiment tracking infrastructure that allows you to track your experiments and loss curves with respect to gradient steps and wallclock time.

## TinyStories

We are going to start with the simple TinyStories dataset we saw before in the tokenization notebook. On this dataset models should train quickly, and we can see some interesting behaviors. Here's an example of a random text sample from TinyStories:

> Once upon a time there was a little boy named Ben. Ben loved to explore the world around him.
He saw many amazing things, like beautiful vases that were on display in a store. One day, Ben was
walking through the store when he came across a very special vase. When Ben saw it he was amazed!
He said, “Wow, that is a really amazing vase! Can I buy it?” The shopkeeper smiled and said, “Of
course you can. You can take it home and show all your friends how amazing it is!” So Ben took the
vase home and he was so proud of it! He called his friends over and showed them the amazing vase.
All his friends thought the vase was beautiful and couldn't believe how lucky Ben was. And that's how
Ben found an amazing vase in the store!

**Hyperparameter tuning:** We'll provide some basic hyperparameters to start with, though you're free to modify them to get better performance:
- `vocab_size` 10000. Typical vocabulary sizes are in the tens to hundreds of thousands. You should vary this and see how the vocabulary and model behavior changes.
- `context_length` 256. Simple datasets such as TinyStories might not need long sequence lengths, but for the later OpenWebText data, you may want to vary this. Try varying this and seeing the impact on both the per-iteration runtime and the final perplexity.
- `d_model` 512. This is slightly smaller than the 768 dimensions used in many small transformer papers, but this will make things faster.
- `d_ff` 1344. This is roughly `(8/3) * d_model` while being a multiple of 64, which is good for GPU performance.
- RoPE $\Theta$ parameter `theta` 10000.
- number of layers and heads 4 layers, 16 heads. Together, this will give about 17M non-embedding parameters which is a fairly small transformer.
- total tokens processed 327,680,000 (your batch size × total step count × context length should equal roughly this value).

It's good to do some trial and error to find good defaults for the following other hyperparameters: learning rate, learning rate warmup, other AdamW hyperparameters $\beta_1$, $\beta_2$, $\varepsilon$), and weight decay. You can find some typical choices of such hyperparameters in *Kingma and Ba (2015)*.

Now we can put everything together by getting a trained BPE tokenizer, tokenizing the training dataset, and running this in the training loop that we've written. 

*Runtime Estimate*: If your implementation is correct and efficient, running this job with the above hyperparameters should take around 30-40 mins on a single H100, though estimates will vary a good bit depending on your hardware. If your runtimes seem unusually long, check and make sure your dataloading, checkpointing, or validation loss code are not bottlenecking your runtimes and that your implementation is properly batched.

*Tip:* We highly recommend getting comfortable with your IDE's built-in debugger (e.g. VSCode/PyCharm), which will save you time compared to debugging with print statements. If you use a text editor, you can use something more like `pdb`. A few other good practices when debugging model architectures are:
- A common first step when developing any neural net architecture is to overfit to a single minibatch. If your implementation is correct, you should be able to quickly drive the training loss to near-zero.
- Set debug breakpoints in various model components, and inspect the shapes of intermediate tensors to make sure they match your expectations.
- Monitor the norms of activations, model weights, and gradients to make sure they are not exploding or vanishing.

#### Problem: Tune the learning rate

The learning rate is one of the most important hyperparameters to tune. Taking the base model you've trained, answer the following questions:
1. Perform a hyperparameter sweep over the learning rates and report the final losses (or note divergence if the optimizer diverges). Generate learning curves associated with multiple learning rates. If possible, try to achieve a per-token validation loss on TinyStories of `1.45` or less.
2. Folk wisdom is that the best learning rate is "at the edge of stability". Investigate how the point at which learning rates diverge is related to your best learning rate. Show the learning curves of increasing learning rate which include at least one divergent run and an analysis of how this relates to convergence rates.

*Runtime Estimate:* Should take ~4 hours to run on a single H100, but this can vary a lot depending on your hardware.

*Tip:* If you are running on CPU or MPS, you should instead reduce the total tokens processed count to 40, 000, 000, which will be sufficient to produce reasonably fluent text. You may also increase the target validation loss. Running our code with a tuned learning rate on an M3 Max chip and 36 GB of RAM, we used `batch size × total step count × context length` of `32 * 5000 * 256 = 40_960_000` tokens, this run took 1 hour and 22 minutes the CPU and 36 minutes on MPS. At step 5000, we achieved a validation loss of `1.80`.

Some additional tips:
- When using $N$ training steps, we suggest adjusting the cosine learning rate decay schedule to terminate its decay, i.e. reach the minimum learning rate, at precisely step $N$.
- When using MPS, do not use TF32 kernels, i.e. do not set `torch.set_float32_matmul_precision('high')` as you might with CUDA devices. We tried enabling TF32 kernels with MPS and found the backend will use silently broken kernels that cause unstable
training.
- You can speed up training by JIT-compiling your model with `torch.compile`:
    - On CPU, compile your model with `model = torch.compile(model)`.
    - On MPS, you can somewhat optimize the backward pass using `model = torch.compile(model, backend='aot_eager')`, but compilation with Inductor was currently not supported on MPS at the time this was written.

In [ ]:
# implement code or implement and call a script

Now let's vary the batch size and see what happens to training. Batch sizes are important. They let us get higher efficiency from our GPUs by doing larger matrix multiplies, but is it true that we always want batch sizes to be large? Let's run some experiments to find out.

#### Problem: Batch size variations

Vary your batch size all the way from 1 to the GPU memory limit. Try at least a few batch sizes in between, including typical sizes like 64 and 128. Plot the learning curves for runs with different batch sizes. The learning rates should be optimized again if necessary.

*Runtime Estimate:* Should take ~2 hours to run on a single H100, but this can vary a lot depending on your hardware.

In [ ]:
# implement code or implement and call a script

With your decoder in hand, we can now generate text! We will generate from the model and see how good it is. As a reference, you should get outputs that look at least as good as the example below.

**Example: Sample output from a TinyStories language model**

> Once upon a time, there was a pretty girl named Lily. She loved to eat gum, especially the big black one. One day, Lily's mom asked her to help cook dinner. Lily was so excited! She loved to help her mom. Lily's mom made a big pot of soup for dinner. Lily was so happy and said, “Thank you, Mommy! I love you.” She helped her mom pour the soup into a big bowl. After dinner, Lily's mom made some yummy soup. Lily loved it! She said, “Thank you, Mommy! This soup is so yummy!” Her mom smiled and said, “I'm glad you like it, Lily.” They finished cooking and continued to cook together. The end.

*Tip:* If you used the low-resource configuration with 40M tokens processed, you should see generations that still resemble English but are not as fluent as above. For example, our sample output from a TinyStories language model trained on 40M tokens is below:

> Once upon a time, there was a little girl named Sue. Sue had a tooth that she loved very much. It was his best head. One day, Sue went for a walk and met a ladybug! They became good friends and played on the path together. “Hey, Polly! Let's go out!” said Tim. Sue looked at the sky and saw that it was difficult to find a way to dance shining. She smiled and agreed to help the talking!” As Sue watched the sky moved, what it was. She

Here is the precise problem statement:

#### Problem: Generate text

Using your decoder and your trained checkpoint, report the text generated by your model. You may need to manipulate the decoder parameters (temperature, top-p, etc.) to get fluent outputs. Generate a text dump of at least 256 tokens of text, or until the first `<|endoftext|>` token. How fluent does the output seem?

In [ ]:
# implement code or implement and call a script

## Ablations and architecture modification

The best way to understand the transformer is to actually modify it and see how it behaves. We will now do a few simple ablations and modifications.

**Ablation 1: layer normalization** It is often said that layer normalization is important for the stability of transformer training. But perhaps we want to live dangerously. Let's remove RMSNorm from each of our transformer blocks and see what happens.

#### Problem: Remove RMSNorm and train

Remove all of the RMSNorms from your transformer and train. What happens at the previous optimal learning rate? Can you get stability by using a lower learning rate? Plot and compare the learning curve for when you remove RMSNorms and train with the learning curve for the best learning rate. What is the impact of the RMSNorm?

*Runtime Estimate:* Should take ~1 hour to run on a single H100, but this can vary a lot depending on your hardware.

In [ ]:
# implement code or implement and call a script

Let's now investigate another layer normalization choice that seems arbitrary at first glance. Pre-norm transformer blocks are defined as

$$
\begin{align*}
z &= x + \text{MultiHeadedSelfAttention}(\text{RMSNorm}(x)) \\
y &= z + \text{FFN}(\text{RMSNorm}(z)) \ .
\end{align*}
$$

This is one of the few "consensus" modifications to the original transformer architecture, which used a post-norm approach as

$$
\begin{align*}
z &= \text{RMSNorm}(x + \text{MultiHeadedSelfAttention}(x)) \\
y &= \text{RMSNorm}(z + \text{FFN}(z)) \ .
\end{align*}
$$

Let's revert back to the post-norm approach and see what happens.

#### Problem: Implement post-norm and train

Modify your pre-norm transformer implementation into a post-norm one. Train with the post-norm model and see what happens. Plot a learning curve for a post-norm transformer, compared to the pre-norm one.

In [ ]:
# implement code or implement and call a script

We see that layer normalization has a major impact on the behavior of the transformer, and that even the position of the layer normalization is important.

**Ablation 2: position embeddings** We will next investigate the impact of the position embeddings on the performance of the model. Specifically, we will compare our base model (with RoPE) with not including position embeddings at all (NoPE). It turns out that decoder-only transformers, i.e. those with a causal mask as we have implemented, can in theory infer relative or absolute position information without being provided with position embeddings explicitly (*Tsai et al., 2019, Kazemnejad et al., 2023*). We will now test empirically how NoPE performs compare to RoPE.

#### Problem: Implement NoPE

Modify your transformer implementation with RoPE to remove the position embedding information entirely, and see what happens. Plot a learning curve comparing the performance of RoPE and NoPE.

*Runtime Estimate:* Should take ~1 hour to run on a single H100, but this can vary a lot depending on your hardware.

In [ ]:
# implement code or implement and call a script

**Ablation 3: SwiGLU vs. SiLU** Next, we will follow *Shazeer (2020)* and test the importance of gating in the feed-forward network, by comparing the performance of SwiGLU feed-forward networks versus feedforward networks using SiLU activations but no gated linear unit (GLU):

$$
\text{FFN}_{\text{SiLU}}(x) = W_2 \text{SiLU}(W_1 x) \ .
$$

Recall that in our SwiGLU implementation, we set the dimensionality of the inner feed-forward layer to be roughly $d_{\mathrm{ff}} = (8/3) d_{\mathrm{model}}$, while ensuring $d_{\mathrm{ff}} \text{ mod } 64 = 0$ to make use of GPU tensor cores. In your $\text{FFN}_{\text{SiLU}}$ implementation you should set $d_{\mathrm{ff}} = 4 d_{\mathrm{model}}$, to approximately match the parameter count of the SwiGLU feed-forward network (which has three instead of two weight matrices).

#### Problem: SwiGLU vs SiLU

Plot a learning curve comparing the performance of SwiGLU and SiLU feed-forward networks, with approximately matched parameter counts.

In [ ]:
# implement code or implement and call a script

*Tip:* If hardware-constrained, consider using smaller datasets like the TinyStories dataset.

In the remainder of the notebook, we will move to the larger-scale, noisier OpenWebText dataset and experiment with architecture modifications there. It takes a long time to train an LM to fluency on OpenWebText, so we suggest those with limited hardware continue testing modifications on TinyStories (using validation loss as a metric to evaluate performance).

## Running on OpenWebText

We will now move to a more standard pretraining dataset created from a web crawl, namely the OpenWebText sample we saw before in the tokenization notebook. Here is an example from OpenWebText. Note how the text is much more realistic, complex, and varied. You may want to look through the training dataset to get a sense of what training data looks like for a web-scraped corpus.

> Baseball Prospectus director of technology Harry Pavlidis took a risk when he hired Jonathan Judge. Pavlidis knew that, as Alan Schwarz wrote in The Numbers Game, “no corner of American culture is more precisely counted, more passionately quantified, than performances of baseball players.” With a few clicks here and there, you can findout that Noah Syndergaard's fastball revolves more than 2,100 times per minute on its way to the plate, that Nelson Cruz had the game's highest average exit velocity among qualified hitters in 2016 and myriad other tidbits that seem ripped from a video game or science fiction novel. The rising ocean of data has empowered an increasingly important actor in baseball's culture: the analytical hobbyist. That empowerment comes with added scrutiny – on the measurements, but also on the people and publications behind them. With Baseball Prospectus, Pavlidis knew all about the backlash that accompanies quantitative imperfection. He also knew the site's catching metrics needed to be reworked, and that it would take a learned mind – someone who could tackle complex statistical modeling problems – to complete the job. “He freaks us out.” Harry Pavlidis Pavlidis had a hunch that Judge “got it” based on the latter's writing and their interaction at a sitesponsored ballpark event. Soon thereafter, the two talked over drinks. Pavlidis' intuition was validated. Judge was a fit for the position – better yet, he was a willing fit. “I spoke to a lot of people,” Pavlidis said, “he was the only one brave enough to take it on.” [...]

*Note:* You may have to re-tune your hyperparameters such as learning rate or batch size for this experiment.

#### Problem: Experiment on OWT

Train your language model on OpenWebText with the same model architecture and total training iterations as TinyStories. How well does this model do? Plot a learning curve of your language model if possible. What is the difference in losses from TinyStories, and how should we interpret these losses? Generate some text from OpenWebText LM in the same format as the TinyStories outputs. How is the fluency of this text? Why is the output quality worse even though we have the same model and compute budget as TinyStories?

*Runtime Estimate:* Should take ~3 hours to run on a single H100, but this can vary a lot depending on your hardware.

In [ ]:
# implement code or implement and call a script